In [0]:
%run ../00-common/01.environment-config

In [0]:
%run ../00-common/02.bronze-helpers

In [0]:
source_file = f"{landing_folder_path}/drivers.json"
table_name = f"{catalog_name}.{bronze_schema}.drivers"

In [0]:
from pyspark.sql.types import StructType, StructField, StringType, DateType

# Define schema for nested name field
name_schema = StructType([
    StructField("familyName", StringType(), True),
    StructField("givenName", StringType(), True)
])

# Define schema for drivers JSON file with StructType and StructField
drivers_schema = StructType([
    StructField("driverId", StringType(), True),
    StructField("dateOfBirth", DateType(), True),
    StructField("name", name_schema, True),
    StructField("nationality", StringType(), True),
    StructField("url", StringType(), True)
])

# Read drivers JSON file from volume with defined schema
drivers_df = (spark.read
    .schema(drivers_schema)
    .option("mode", "FAILFAST")
    .json(source_file)
)

#display(drivers_df)

In [0]:
# Add ingestion timestamp and source file path metadata using helper function
drivers_with_metadata_df = add_ingestion_metadata(drivers_df)

# display(drivers_with_metadata_df)

In [0]:
# Write drivers data to bronze layer table
(drivers_with_metadata_df.write
    .mode("overwrite")
    .format("delta")
    .saveAsTable(table_name)
)

print(f"✅ Drivers data successfully written to {table_name} table")

In [0]:
%sql
-- SELECT 
--   driverId,
--   dateOfBirth,
--   name.familyName,
--   name.givenName,
--   nationality,
--   url,
--   ingestion_timestamp,
--   source_file_path
-- FROM formula1.bronze.drivers
-- LIMIT 10